## 2020年Q1-2026年Q6新能源电动汽车销量

In [22]:
import numpy as np, pandas as pd
from pyecharts import options as opts
from pyecharts.charts import Line, Pie, Bar

df = pd.read_excel('中国汽车行业多维销量数据库.xlsx', sheet_name='EV车型销量_20Q1-26Q6_16888')
df.head()

,年份,月份,厂商,车型,售价（万元）,销量,数据来源
0,2020,1,华晨宝马,宝马5系,36.80 - 52.59,12313,汽车之家(16888.com)电动车销量榜单
1,2020,1,北京奔驰,奔驰C级,29.99 - 34.56,12300,汽车之家(16888.com)电动车销量榜单
2,2020,1,沃尔沃亚太,沃尔沃XC60,39.69 - 47.49,6684,汽车之家(16888.com)电动车销量榜单
3,2020,1,长安汽车,逸动,7.29 - 9.99,4593,汽车之家(16888.com)电动车销量榜单
4,2020,1,上汽集团,荣威Ei5,0.00 - 0.00,3068,汽车之家(16888.com)电动车销量榜单


In [23]:
# 转为 str 类型
df['年份'] = df['年份'].astype("str")
df['月份'] = df['月份'].astype("str")


### 1. 中国电动汽车月销售量

In [24]:
month_sales = df.groupby(['年份', '月份'])['销量'].sum().reset_index()
month_sales['日期'] = pd.to_datetime(
    month_sales['年份'] + '-' + month_sales['月份']).dt.strftime("%Y-%m")  # 截取 年-月
month_sales = month_sales.sort_values('日期')

sales_line = (
    Line()
    .add_xaxis(month_sales['日期'].to_list())
    .add_yaxis("销量", month_sales['销量'].to_list(), label_opts=opts.LabelOpts(is_show=False))
    .set_global_opts(title_opts=opts.TitleOpts(title="电动汽车月销售量"),
                     xaxis_opts=opts.AxisOpts(name="月份"),
                     yaxis_opts=opts.AxisOpts(name="销量"))
)
sales_line.render_notebook()

* 20年至26年上半年，电动汽车销量总体保持上涨趋势。25年12月达到峰值1,379,570辆，创历史新高；25年下半年起月度销量稳定在130万辆以上。

* 每年1、2月是销量淡季。造成这样现象的原因是，按照现有市场规律，上一年度销量目标已经完成，各大经销商销售压力降低，促销力度减小。同时需求端提前在上一年度年底提前释放。

* 9-12月是全年销量最集中的几个月份，年底受市场需求和各经销商销售业绩考核影响，普遍是上涨趋势。26年上半年虽未到年末冲量期，但3月起已明显回升，月销重新站上100万辆。


### 2. 中国汽车年销售情况

In [25]:
years_sales = df.groupby('年份')['销量'].sum().reset_index()
# 前后两项 相减
years_sales['销量'].diff()
# 向下移动一位，错位
years_sales['销量'].shift(1, axis=0)
# 同比计算
years_sales['同比'] = round(
    (years_sales['销量'].diff() / years_sales['销量'].shift(1)) * 100, 2)
years_sales

,年份,销量,同比
0,2020,1691581,NaN
1,2021,3528822,108.61
2,2022,6860452,94.41
3,2023,8226826,19.92
4,2024,11390080,38.45
5,2025,13380003,17.47
6,2026,4928504,-63.17


In [26]:
year_bar = (
    Bar()
    .add_xaxis(years_sales['年份'].astype('str').to_list())
    .add_yaxis("销量", years_sales['销量'].to_list(), label_opts=opts.LabelOpts(is_show=True), color='red',
    #  z=0
     )
    # 设置次坐标轴
    .extend_axis(
        yaxis=opts.AxisOpts(
            name="同步%",
            type_="value",
        )
    ).set_global_opts(title_opts=opts.TitleOpts(title="中国电动汽车年销售量情况"),
                      yaxis_opts=opts.AxisOpts(name="销量"))
)

years_line = (
    Line()
    .add_xaxis(years_sales['年份'].astype('str').to_list())
    .add_yaxis("同比", years_sales['同比'].to_list(), label_opts=opts.LabelOpts(is_show=False), yaxis_index=1, color='blue')
)
year_bar.overlap(years_line).render_notebook()

In [27]:
# 2022年相当2020年销量同步增长率
sales_2022 = years_sales.iloc[2]["销量"]
sales_2020 = years_sales.iloc[0]["销量"]
sales_per_20_22 = round((sales_2022 - sales_2020) / sales_2020, 2)
sales_per_20_22

np.float64(3.06)

* 20年-25年电动汽车销量持续增长：20年169.2万辆 → 21年352.9万辆（+108.6%）→ 22年686.0万辆（+94.4%）→ 23年822.7万辆（+19.9%）→ 24年1,139.0万辆（+38.5%）→ 25年1,338.0万辆（+17.5%）。
* 21年同比增幅最大，此后增速总体回落；24年在以旧换新补贴和密集新车上市带动下阶段性回升。22年是行业放量关键年，销量约为20年的4.1倍（增长率305.6%）。
* 26年上半年销量492.9万辆，同比继续增长，全年有望再创新高。


### 3. 销量前20的车型

In [28]:
top20_model = df.groupby(['车型', '售价（万元）', '厂商'])['销量'].sum(
).reset_index().sort_values('销量', ascending=False).reset_index(drop=True).head(20)

In [29]:
top20_model_pie = (
    Pie()
    .add("", [list(z) for z in zip(top20_model['车型'].to_list(), top20_model['销量'].to_list())])
    .set_colors(["MediumOrchid", "green", "GreenYellow", "red", "pink", "orange", "DarkKhaki", "LightBlue", "DarkOrange", "Magenta"])
    .set_global_opts(title_opts=opts.TitleOpts(title="近9年销量前20的车型", 
                    #  is_show=False
                     ),
                     legend_opts=opts.LegendOpts(type_="scroll", pos_right="0", orient="vertical"),)
    .set_series_opts(label_opts=opts.LabelOpts(formatter="{b}：{d}%"))
)
top20_model_pie.render_notebook()

In [30]:
# 销量前20的车最低售价情况
top20_model['最低售价（万元）'] = pd.to_numeric(top20_model['售价（万元）'].apply(lambda x: x.split('-')[0]))
top20_model.sort_values('最低售价（万元）', ascending=False)

,车型,售价（万元）,厂商,销量,最低售价（万元）
11,宝马5系,36.80 - 52.59,华晨宝马,867769,36.80
10,奔驰C级,29.99 - 34.56,北京奔驰,896907,29.99
0,Model Y,26.35 - 31.35,特斯拉中国,2188134,26.35
4,Model 3,23.55 - 33.95,特斯拉中国,1154618,23.55
16,唐新能源,17.98 - 19.98,比亚迪,586201,17.98
6,汉,16.58 - 23.58,比亚迪,1058621,16.58
14,AION S,13.68 - 13.98,广汽埃安,660009,13.68
8,元PLUS,11.58 - 14.99,比亚迪,947715,11.58
12,宋Pro新能源,10.28 - 13.38,比亚迪,738920,10.28
9,海豚,9.98 - 12.98,比亚迪,925175,9.98


* 20年-26年上半年累计，Model Y（218.8万辆）与宏光MINIEV（211.5万辆）销量几乎并列第一，是仅有的两款累计超200万辆的车型；秦PLUS、宋PLUS新能源分列三、四位。
* 比亚迪是前20车型中的最大赢家，共9款车型上榜，覆盖代步、家用、高端全价位；特斯拉凭Model Y/Model 3两款车型挤进前五。
* 售价20万以上的有4个车型，10万以下的有11个车型——性价比车型仍是市场绝对主力。


In [31]:

# 销量前20的车型对应的车企
top20_model_company = top20_model.groupby(['厂商'])['销量'].sum().reset_index().sort_values('销量', ascending=False)


top20_model_company_pie = (
    Pie()
    .add("", [list(z) for z in zip(top20_model_company['厂商'].to_list(), top20_model_company['销量'].to_list())])
    .set_colors(["MediumOrchid", "green", "GreenYellow", "red", "pink", "orange", "purple", "LightBlue", "DarkOrange", "Magenta"])
    .set_global_opts(legend_opts=opts.LegendOpts(type_="scroll", pos_right="0", orient="vertical"))
    .set_series_opts(label_opts=opts.LabelOpts(formatter="{b}：{d}%"))
)
top20_model_company_pie.render_notebook()

* 从上图表，销量前20的车型对应的车企，比亚迪以9款车型占了44.4%，特斯拉两款车型占16.5%，上汽通用五菱凭宏光MINIEV和缤果占13.0%，长安汽车占7.4%。头部集中度极高，比亚迪一家接近半壁江山。
* 新势力（理想、零跑等）与自主品牌（长安、吉利、广汽埃安）车型陆续进入前20，合资品牌仅剩奔驰、宝马的燃油时代明星车型在列。



### 4. 销量前20的车企

In [32]:

top20_company = df.groupby('厂商')['销量'].sum().reset_index(
).sort_values('销量', ascending=False).head(20)

In [33]:

top20_company_pie = (
    Pie()
    .add("", [list(z) for z in zip(top20_company['厂商'].to_list(), top20_company['销量'].to_list())])
    .set_colors(["MediumOrchid", "green", "GreenYellow", "red", "pink", "orange", "DarkKhaki", "LightBlue", "DarkOrange", "Magenta"])
    .set_global_opts(title_opts=opts.TitleOpts(title="", 
                                            #    is_show=False
                                               ),
                     legend_opts=opts.LegendOpts(type_="scroll", pos_right="0", orient="vertical"),)
    .set_series_opts(label_opts=opts.LabelOpts(formatter="{b}: {c}"))
)
top20_company_pie.render_notebook()

* 20年-26年上半年各大厂商累计销量，比亚迪以1,243.6万辆遥遥领先，是第二名上汽通用五菱（342.3万辆）的3.6倍。五菱凭借宏光MINIEV和缤果反超特斯拉（334.3万辆），升至第二，特斯拉滑落至第三。
* 吉利（219.7万辆）、长安（192.0万辆）、理想（173.1万辆）分列4-6位，理想仍是新势力中表现最好的；零跑（136.8万辆）、AITO问界（115.8万辆）也进入前十。
* 华晨宝马是唯一进入前十的合资品牌。新能源格局已从早期的"一超两强"演变为"比亚迪一家独大 + 自主/新势力集体上位"，传统合资品牌在榜单中加速边缘化。



### 5. 2023年前10厂商汽车车型销量分布

In [34]:

df_2023 = df[df["年份"] == "2023"]
# 2023年前10销量的厂商
df_2023_top10_company = df_2023.groupby('厂商')['销量'].sum().reset_index(
).sort_values('销量', ascending=False).head(10)
df_2023_top10_company

,厂商,销量
69,比亚迪,2568956
75,特斯拉中国,603664
58,广汽埃安,477545
16,上汽通用五菱,459971
91,长安汽车,377845
76,理想,371923
38,华晨宝马,218582
43,吉利汽车,198601
32,北京奔驰,195835
81,蔚来,159888


In [35]:

# 2023年前10销量的厂商对应的车型
df_2023_top10_company_model = df_2023[df_2023['厂商'].isin(
    df_2023_top10_company['厂商'])]

In [36]:

# 内外图表
outer_data = df_2023_top10_company_model.groupby(['厂商', '车型'])['销量'].sum().reset_index().sort_values('厂商', ascending=False)
inter_data = outer_data.groupby('厂商')['销量'].sum().reset_index().sort_values('厂商', ascending=False)

In [37]:

(
    Pie()
    .add(
        series_name="销量",
        data_pair=[list(z) for z in zip(inter_data['厂商'], inter_data['销量'])],
        radius=[0, "50%"],
        label_opts=opts.LabelOpts(position="inner"),
    )
    .add(
        series_name="销量",
        radius=["50%", "80%"],
        data_pair=[list(z) for z in zip(outer_data['车型'], outer_data['销量'])],
    )
    .set_colors(["MediumOrchid", "green", "GreenYellow", "red", "pink", "orange", "DarkKhaki", "LightBlue", "DarkOrange", "Magenta"])
    .set_global_opts(legend_opts=opts.LegendOpts(pos_left="left", orient="vertical", is_show=False))
    .set_series_opts(
        tooltip_opts=opts.TooltipOpts(
            trigger="item", formatter="{a} <br/>{b}: {c} ({d}%)"
        )
    )
    .render_notebook()
)